In [1]:
# STEP 4.1 – Import Libraries

import pandas as pd
import numpy as np

# Train-test split
from sklearn.model_selection import train_test_split

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Regression models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor

# Clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_squared_error,
    r2_score
)

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Model saving
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


In [2]:
# STEP 4.2 – Load Model-Ready Data

X_cls = pd.read_csv('../data/X_classification.csv')
y_cls = pd.read_csv('../data/y_classification.csv')

X_reg = pd.read_csv('../data/X_regression.csv')
y_reg = pd.read_csv('../data/y_regression.csv')

X_cluster = pd.read_csv('../data/X_clustering.csv')

print("Classification:", X_cls.shape, y_cls.shape)
print("Regression    :", X_reg.shape, y_reg.shape)
print("Clustering    :", X_cluster.shape)


Classification: (132379, 18) (132379, 1)
Regression    : (132379, 18) (132379, 1)
Clustering    : (132379, 18)


In [3]:
# STEP 4.3 – Train-Test Split
# Classification Split

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

print("Classification Train:", X_train_cls.shape)
print("Classification Test :", X_test_cls.shape)


Classification Train: (105903, 18)
Classification Test : (26476, 18)


In [4]:
# Regression Split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=42
)

print("Regression Train:", X_train_reg.shape)
print("Regression Test :", X_test_reg.shape)


Regression Train: (105903, 18)
Regression Test : (26476, 18)


In [5]:
# STEP 4.4 – Handle Class Imbalance (SMOTE – TRAIN DATA ONLY)

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_cls, y_train_cls
)

print("Before SMOTE:\n", y_train_cls.value_counts())
print("\nAfter SMOTE:\n", y_train_smote.value_counts())


Before SMOTE:
 converted
0            83060
1            22843
Name: count, dtype: int64

After SMOTE:
 converted
0            83060
1            83060
Name: count, dtype: int64


In [6]:
# STEP 4.5 – CLASSIFICATION MODELS
# 4.5.1 Logistic Regression (Baseline)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_smote, y_train_smote)

y_pred_lr = lr.predict(X_test_cls)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test_cls, y_pred_lr))
print(classification_report(y_test_cls, y_pred_lr))


Logistic Regression Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20765
           1       1.00      1.00      1.00      5711

    accuracy                           1.00     26476
   macro avg       1.00      1.00      1.00     26476
weighted avg       1.00      1.00      1.00     26476



In [7]:
# 4.5.2 Decision Tree

dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train_smote, y_train_smote)

y_pred_dt = dt.predict(X_test_cls)

print("Decision Tree Accuracy:",
      accuracy_score(y_test_cls, y_pred_dt))


Decision Tree Accuracy: 1.0


In [8]:
# 4.5.3 Random Forest (FINAL CLASSIFIER)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42
)

rf.fit(X_train_smote, y_train_smote)

y_pred_rf = rf.predict(X_test_cls)

print("Random Forest Accuracy:",
      accuracy_score(y_test_cls, y_pred_rf))
print(classification_report(y_test_cls, y_pred_rf))


Random Forest Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20765
           1       1.00      1.00      1.00      5711

    accuracy                           1.00     26476
   macro avg       1.00      1.00      1.00     26476
weighted avg       1.00      1.00      1.00     26476



In [9]:
# STEP 4.6 – REGRESSION MODELS
# 4.6.1 Linear Regression

lr_reg = LinearRegression()
lr_reg.fit(X_train_reg, y_train_reg)

y_pred_lr = lr_reg.predict(X_test_reg)

print("Linear Regression RMSE:",
      np.sqrt(mean_squared_error(y_test_reg, y_pred_lr)))
print("R2 Score:", r2_score(y_test_reg, y_pred_lr))


Linear Regression RMSE: 207.23198501243215
R2 Score: 0.9295589638152899


In [10]:
# 4.6.2 Gradient Boosting Regressor

gbr = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gbr.fit(X_train_reg, y_train_reg)

y_pred_gbr = gbr.predict(X_test_reg)

print("GBR RMSE:",
      np.sqrt(mean_squared_error(y_test_reg, y_pred_gbr)))
print("GBR R2:",
      r2_score(y_test_reg, y_pred_gbr))


GBR RMSE: 24.149602395025635
GBR R2: 0.9990433961345404


In [11]:
# STEP 4.7 – CLUSTERING (Unsupervised)

kmeans = KMeans(n_clusters=4, random_state=42)
clusters = kmeans.fit_predict(X_cluster)

sil_score = silhouette_score(X_cluster, clusters)

print("Silhouette Score:", sil_score)


Silhouette Score: 0.14212654136983757


In [12]:
# STEP 4.8 – Create Models Folder

os.makedirs('../models', exist_ok=True)
print("✅ models directory ready")


✅ models directory ready


In [13]:
# STEP 4.9 – Save Final Models

joblib.dump(rf, '../models/classification_model.pkl')
joblib.dump(gbr, '../models/regression_model.pkl')
joblib.dump(kmeans, '../models/clustering_model.pkl')

print("✅ All models saved successfully")


✅ All models saved successfully
